# Silver para Gold

Cria a base usada nas análises e separa as partidas com virada.

In [0]:
from pyspark.sql import functions as F

partidas = spark.table("workspace.silver.partidas")
gols = spark.table("workspace.silver.gols")
estatisticas = spark.table("workspace.silver.estatisticas")

## Estatísticas por partida

In [0]:
estatisticas_mandante = estatisticas.select(
    "partida_id",
    F.col("clube").alias("clube_mandante_estatistica"),
    F.col("posse_bola_pct").alias("posse_mandante"),
    F.col("escanteios").alias("escanteios_mandante"),
    F.col("chutes").alias("chutes_mandante"),
    F.col("chutes_no_alvo").alias("chutes_alvo_mandante"),
    F.col("estatisticas_disponiveis").alias("estatisticas_mandante_disponiveis")
)

estatisticas_visitante = estatisticas.select(
    "partida_id",
    F.col("clube").alias("clube_visitante_estatistica"),
    F.col("posse_bola_pct").alias("posse_visitante"),
    F.col("escanteios").alias("escanteios_visitante"),
    F.col("chutes").alias("chutes_visitante"),
    F.col("chutes_no_alvo").alias("chutes_alvo_visitante"),
    F.col("estatisticas_disponiveis").alias("estatisticas_visitante_disponiveis")
)

## Primeiro gol

In [0]:
primeiro_minuto = gols.groupBy("partida_id").agg(
    F.min("minuto_ordem").alias("primeiro_minuto")
)

primeiros_gols = (
    gols.join(primeiro_minuto, "partida_id")
    .filter(F.col("minuto_ordem") == F.col("primeiro_minuto"))
    .groupBy("partida_id", "primeiro_minuto")
    .agg(F.collect_set("clube").alias("clubes_primeiro_gol"))
    .withColumn(
        "primeiro_clube",
        F.when(F.size("clubes_primeiro_gol") == 1, F.element_at("clubes_primeiro_gol", 1))
    )
    .withColumn("primeiro_gol_ambiguo", F.size("clubes_primeiro_gol") != 1)
    .drop("clubes_primeiro_gol")
)

## Tabela analítica

In [0]:
base = (
    partidas.alias("p")
    .join(
        estatisticas_mandante.alias("em"),
        (F.col("p.partida_id") == F.col("em.partida_id")) &
        (F.col("p.mandante") == F.col("em.clube_mandante_estatistica")),
        "left"
    )
    .join(
        estatisticas_visitante.alias("ev"),
        (F.col("p.partida_id") == F.col("ev.partida_id")) &
        (F.col("p.visitante") == F.col("ev.clube_visitante_estatistica")),
        "left"
    )
    .join(primeiros_gols.alias("pg"), F.col("p.partida_id") == F.col("pg.partida_id"), "left")
    .select(
        F.col("p.partida_id"),
        F.year("p.data_partida").alias("ano"),
        F.col("p.data_partida"),
        F.col("p.rodada"),
        F.col("p.mandante"),
        F.col("p.visitante"),
        F.col("em.clube_mandante_estatistica"),
        F.col("ev.clube_visitante_estatistica"),
        F.col("p.estado_mandante"),
        F.col("p.estado_visitante"),
        F.col("p.placar_mandante"),
        F.col("p.placar_visitante"),
        F.col("p.vencedor"),
        F.col("p.resultado"),
        F.col("em.posse_mandante"),
        F.col("ev.posse_visitante"),
        F.col("em.escanteios_mandante"),
        F.col("ev.escanteios_visitante"),
        F.col("em.chutes_mandante"),
        F.col("ev.chutes_visitante"),
        F.col("em.chutes_alvo_mandante"),
        F.col("ev.chutes_alvo_visitante"),
        F.col("em.estatisticas_mandante_disponiveis"),
        F.col("ev.estatisticas_visitante_disponiveis"),
        F.col("pg.primeiro_clube"),
        F.col("pg.primeiro_minuto"),
        F.coalesce(F.col("pg.primeiro_gol_ambiguo"), F.lit(False)).alias("primeiro_gol_ambiguo")
    )
)

In [0]:
partidas_analiticas = (
    base.withColumn(
        "primeiro_gol_venceu",
        F.when(F.col("primeiro_clube").isNull(), F.lit(None).cast("boolean"))
         .otherwise(F.col("primeiro_clube") == F.col("vencedor"))
    )
    .withColumn(
        "houve_virada",
        F.when(F.col("primeiro_clube").isNull(), F.lit(False))
         .otherwise((F.col("resultado") != "empate") & (F.col("primeiro_clube") != F.col("vencedor")))
    )
    .withColumn(
        "estado_vencedor",
        F.when(F.col("resultado") == "mandante", F.col("estado_mandante"))
         .when(F.col("resultado") == "visitante", F.col("estado_visitante"))
    )
    .withColumn(
        "estatisticas_disponiveis",
        F.coalesce(F.col("estatisticas_mandante_disponiveis"), F.lit(False)) &
        F.coalesce(F.col("estatisticas_visitante_disponiveis"), F.lit(False))
    )
    .withColumn("processado_em", F.current_timestamp())
    .drop("clube_mandante_estatistica", "clube_visitante_estatistica")
)

viradas = partidas_analiticas.filter("houve_virada")

## Validação

In [0]:
total_silver = partidas.count()
total_gold = partidas_analiticas.count()
duplicadas = partidas_analiticas.groupBy("partida_id").count().filter("count > 1").count()
times_estatisticas_incorretos = base.filter(
    (F.col("clube_mandante_estatistica") != F.col("mandante")) |
    (F.col("clube_visitante_estatistica") != F.col("visitante"))
).count()

validacao = [
    ("Partidas na Silver", total_silver),
    ("Partidas na Gold", total_gold),
    ("IDs duplicados", duplicadas),
    ("Clubes de estatísticas incorretos", times_estatisticas_incorretos),
    ("Viradas identificadas", viradas.count()),
]

display(spark.createDataFrame(validacao, ["medida", "valor"]))

if total_silver != total_gold or duplicadas > 0 or times_estatisticas_incorretos > 0:
    raise ValueError("Falha na validação da Gold")

medida,valor
Partidas na Silver,9165
Partidas na Gold,9165
IDs duplicados,0
Clubes de estatísticas incorretos,0
Viradas identificadas,418


## Gravação

In [0]:
(partidas_analiticas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.partidas_analiticas"))

(viradas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.viradas"))

In [0]:
display(spark.sql("SHOW TABLES IN workspace.gold"))
display(partidas_analiticas.orderBy("partida_id").limit(10))

database,tableName,isTemporary
gold,partidas_analiticas,false
gold,viradas,false


partida_id,ano,data_partida,rodada,mandante,visitante,estado_mandante,estado_visitante,placar_mandante,placar_visitante,vencedor,resultado,posse_mandante,posse_visitante,escanteios_mandante,escanteios_visitante,chutes_mandante,chutes_visitante,chutes_alvo_mandante,chutes_alvo_visitante,estatisticas_mandante_disponiveis,estatisticas_visitante_disponiveis,primeiro_clube,primeiro_minuto,primeiro_gol_ambiguo,primeiro_gol_venceu,houve_virada,estado_vencedor,estatisticas_disponiveis,processado_em
1,2003,2003-03-29,1,Guarani,Vasco,SP,RJ,4,2,Guarani,mandante,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,SP,false,2026-09-22T23:16:47.653Z
2,2003,2003-03-29,1,Athletico-PR,Gremio,PR,RS,2,0,Athletico-PR,mandante,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,PR,false,2026-09-22T23:16:47.653Z
3,2003,2003-03-30,1,Flamengo,Coritiba,RJ,PR,1,1,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
4,2003,2003-03-30,1,Goias,Paysandu,GO,PA,2,2,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
5,2003,2003-03-30,1,Internacional,Ponte Preta,RS,SP,1,1,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
6,2003,2003-03-30,1,Criciuma,Fluminense,SC,RJ,2,0,Criciuma,mandante,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,SC,false,2026-09-22T23:16:47.653Z
7,2003,2003-03-30,1,Juventude,Sao Paulo,RS,SP,2,2,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
8,2003,2003-03-30,1,Fortaleza,Bahia,CE,BA,0,0,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
9,2003,2003-03-30,1,Cruzeiro,Sao Caetano,MG,SP,2,2,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
10,2003,2003-03-30,1,Vitoria,Figueirense,BA,SC,1,1,-,empate,null,null,0,0,0,0,0,0,false,false,null,null,false,null,false,null,false,2026-09-22T23:16:47.653Z
